In [0]:
# Cell 1: Setup - Create sample manufacturing defect data
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Generate synthetic manufacturing defect data
random.seed(42)
np.random.seed(42)

n = 1000
data = {
    "record_id": range(1, n + 1),
    "machine_id": [f"MACH-{random.randint(1, 5):02d}" for _ in range(n)],
    "product_id": [f"PROD-{random.randint(100, 999)}" for _ in range(n)],
    "shift": [random.choice(["Morning", "Afternoon", "Night", "NIGHT", "morning"]) for _ in range(n)],
    "defect_type": [random.choice(["Scratch", "Dent", "Crack", "None", "none", "CRACK", None]) for _ in range(n)],
    "units_produced": [random.randint(50, 500) for _ in range(n)],
    "units_defective": [random.randint(0, 30) for _ in range(n)],
    "temperature_c": [round(random.uniform(60, 120), 2) for _ in range(n)],
    "timestamp": [datetime(2024, 1, 1) + timedelta(hours=random.randint(0, 8760)) for _ in range(n)]
}

# Introduce duplicates and nulls to simulate raw dirty data
df = pd.DataFrame(data)
df = pd.concat([df, df.sample(50)], ignore_index=True)  # add 50 duplicates
df.loc[df.sample(30).index, "temperature_c"] = None      # add 30 null temps
df.loc[df.sample(20).index, "units_produced"] = None     # add 20 null units

print(f"Raw dataset shape: {df.shape}")
print(f"Nulls:\n{df.isnull().sum()}")
df.head(10)

Raw dataset shape: (1050, 9)
Nulls:
record_id            0
machine_id           0
product_id           0
shift                0
defect_type        160
units_produced      20
units_defective      0
temperature_c       30
timestamp            0
dtype: int64


,record_id,machine_id,product_id,shift,defect_type,units_produced,units_defective,temperature_c,timestamp
0,1,MACH-01,PROD-389,Morning,Scratch,418.0,18,65.11,2024-04-09 08:00:00
1,2,MACH-01,PROD-843,Afternoon,Dent,213.0,30,72.60,2024-01-21 03:00:00
2,3,MACH-03,PROD-405,Night,Dent,188.0,19,104.49,2024-08-05 13:00:00
3,4,MACH-02,PROD-957,NIGHT,None,472.0,2,103.56,2024-10-09 15:00:00
4,5,MACH-02,PROD-701,morning,CRACK,89.0,15,72.70,2024-10-07 17:00:00
5,6,MACH-02,PROD-693,Afternoon,none,342.0,28,114.34,2024-07-26 02:00:00
6,7,MACH-01,PROD-775,NIGHT,None,240.0,6,111.09,2024-10-29 10:00:00
7,8,MACH-05,PROD-601,Morning,Crack,111.0,16,68.55,2024-07-05 21:00:00
8,9,MACH-01,PROD-986,Afternoon,Scratch,307.0,3,71.04,2024-04-13 17:00:00
9,10,MACH-05,PROD-252,NIGHT,Scratch,394.0,27,97.01,2024-05-08 07:00:00


In [0]:
# Cell 2: BRONZE LAYER - Ingest raw data as-is into Delta table
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

# Convert pandas df to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Add ingestion metadata
bronze_df = spark_df.withColumn("ingested_at", current_timestamp()) \
                    .withColumn("source", lit("manufacturing_csv"))

# Write to Delta table (Bronze) - raw, no transformations
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_manufacturing")

print(f"Bronze layer loaded: {bronze_df.count()} rows")
print("Schema:")
bronze_df.printSchema()

Bronze layer loaded: 1050 rows
Schema:
root
 |-- record_id: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- defect_type: string (nullable = true)
 |-- units_produced: double (nullable = true)
 |-- units_defective: long (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = false)
 |-- source: string (nullable = false)



In [0]:
# Cell 3: SILVER LAYER - Clean, deduplicate, standardize
from pyspark.sql.functions import (
    upper, trim, when, col, avg, round as spark_round,
    current_timestamp, lit
)

# Read from Bronze
silver_df = spark.table("bronze_manufacturing")

# 1. Remove duplicates
silver_df = silver_df.dropDuplicates(["record_id"])

# 2. Standardize shift and defect_type to uppercase, trim whitespace
silver_df = silver_df.withColumn("shift", upper(trim(col("shift")))) \
                     .withColumn("defect_type", upper(trim(col("defect_type"))))

# 3. Normalize defect_type - treat "NONE" and nulls as "NO_DEFECT"
silver_df = silver_df.withColumn("defect_type",
    when((col("defect_type") == "NONE") | col("defect_type").isNull(), "NO_DEFECT")
    .otherwise(col("defect_type"))
)

# 4. Fill null temperature_c with average
avg_temp = silver_df.select(avg("temperature_c")).collect()[0][0]
silver_df = silver_df.fillna({"temperature_c": round(avg_temp, 2)})

# 5. Fill null units_produced with median (approx)
avg_units = silver_df.select(avg("units_produced")).collect()[0][0]
silver_df = silver_df.fillna({"units_produced": round(avg_units)})

# 6. Add defect flag
silver_df = silver_df.withColumn("has_defect",
    when(col("defect_type") == "NO_DEFECT", 0).otherwise(1)
)

# 7. Add processing metadata
silver_df = silver_df.withColumn("processed_at", current_timestamp()) \
                     .withColumn("layer", lit("silver"))

# Write Silver Delta table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_manufacturing")

print(f"Silver layer: {silver_df.count()} rows (duplicates removed)")
print(f"\nDefect type distribution:")
silver_df.groupBy("defect_type").count().orderBy("count", ascending=False).show()
print(f"\nShift distribution:")
silver_df.groupBy("shift").count().orderBy("shift").show()

Silver layer: 1000 rows (duplicates removed)

Defect type distribution:
+-----------+-----+
|defect_type|count|
+-----------+-----+
|  NO_DEFECT|  433|
|      CRACK|  291|
|    SCRATCH|  139|
|       DENT|  137|
+-----------+-----+


Shift distribution:
+---------+-----+
|    shift|count|
+---------+-----+
|AFTERNOON|  196|
|  MORNING|  395|
|    NIGHT|  409|
+---------+-----+



In [0]:
# Cell 4: GOLD LAYER - Aggregated KPIs for analytics
from pyspark.sql.functions import (
    count, sum as spark_sum, avg, round as spark_round,
    col, current_timestamp, lit
)

# Read from Silver
gold_input = spark.table("silver_manufacturing")

# --- Gold Table 1: Defect Rate by Machine ---
machine_kpis = gold_input.groupBy("machine_id").agg(
    spark_sum("units_produced").alias("total_units_produced"),
    spark_sum("units_defective").alias("total_units_defective"),
    count("record_id").alias("total_runs"),
    spark_round(avg("temperature_c"), 2).alias("avg_temperature_c"),
    spark_round(
        spark_sum("units_defective") / spark_sum("units_produced") * 100, 2
    ).alias("defect_rate_pct")
).withColumn("reported_at", current_timestamp())

machine_kpis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_machine_kpis")

print("=== GOLD: Machine KPIs ===")
machine_kpis.orderBy("defect_rate_pct", ascending=False).show()

# --- Gold Table 2: Defect Breakdown by Shift ---
shift_defects = gold_input.groupBy("shift", "defect_type").agg(
    count("record_id").alias("occurrences"),
    spark_sum("units_defective").alias("total_defective_units")
).orderBy("shift", "occurrences", ascending=False)

shift_defects.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_shift_defects")

print("=== GOLD: Defects by Shift ===")
shift_defects.show(20)

# --- Gold Table 3: Overall Summary ---
summary = gold_input.agg(
    spark_sum("units_produced").alias("total_units_produced"),
    spark_sum("units_defective").alias("total_defective_units"),
    spark_round(
        spark_sum("units_defective") / spark_sum("units_produced") * 100, 2
    ).alias("overall_defect_rate_pct"),
    spark_round(avg("temperature_c"), 2).alias("avg_temp_c"),
    count("record_id").alias("total_production_runs")
).withColumn("reported_at", current_timestamp())

summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_summary")

print("=== GOLD: Overall Summary ===")
summary.show()

=== GOLD: Machine KPIs ===
+----------+--------------------+---------------------+----------+-----------------+---------------+--------------------+
|machine_id|total_units_produced|total_units_defective|total_runs|avg_temperature_c|defect_rate_pct|         reported_at|
+----------+--------------------+---------------------+----------+-----------------+---------------+--------------------+
|   MACH-03|             51842.0|                 3156|       195|             90.6|           6.09|2026-05-29 06:35:...|
|   MACH-05|             51407.0|                 3110|       198|             89.3|           6.05|2026-05-29 06:35:...|
|   MACH-04|             57051.0|                 3292|       207|            90.38|           5.77|2026-05-29 06:35:...|
|   MACH-02|             56674.0|                 2890|       201|            90.23|            5.1|2026-05-29 06:35:...|
|   MACH-01|             57322.0|                 2919|       199|            91.21|           5.09|2026-05-29 06:35:..

In [0]:
# Cell 5: Pipeline verification - confirm all 3 layers exist
print("=" * 50)
print("MEDALLION PIPELINE - LAYER SUMMARY")
print("=" * 50)

bronze = spark.table("bronze_manufacturing")
silver = spark.table("silver_manufacturing")
gold_machines = spark.table("gold_machine_kpis")
gold_shifts = spark.table("gold_shift_defects")
gold_summary = spark.table("gold_summary")

print(f"\n🥉 BRONZE  | {bronze.count():,} rows | Raw, unmodified")
print(f"🥈 SILVER  | {silver.count():,} rows | Cleaned & deduplicated")
print(f"🥇 GOLD    | {gold_machines.count()} machine KPI rows")
print(f"🥇 GOLD    | {gold_shifts.count()} shift-defect rows")

print("\n=== OVERALL PLANT SUMMARY ===")
gold_summary.select(
    "total_units_produced",
    "total_defective_units",
    "overall_defect_rate_pct",
    "avg_temp_c",
    "total_production_runs"
).show()

print("\nAll Delta tables registered:")
spark.sql("SHOW TABLES").show()

MEDALLION PIPELINE - LAYER SUMMARY

🥉 BRONZE  | 1,050 rows | Raw, unmodified
🥈 SILVER  | 1,000 rows | Cleaned & deduplicated
🥇 GOLD    | 5 machine KPI rows
🥇 GOLD    | 12 shift-defect rows

=== OVERALL PLANT SUMMARY ===
+--------------------+---------------------+-----------------------+----------+---------------------+
|total_units_produced|total_defective_units|overall_defect_rate_pct|avg_temp_c|total_production_runs|
+--------------------+---------------------+-----------------------+----------+---------------------+
|            274296.0|                15367|                    5.6|     90.34|                 1000|
+--------------------+---------------------+-----------------------+----------+---------------------+


All Delta tables registered:
+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|bronze_manufacturing|      false|
| default|   gold_machine_kpis|      false|
| default|  gold_sh